In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  
api_key = os.getenv("OPENROUTER_API_KEY")

model = os.getenv("MODEL")

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openrouter import ChatOpenRouter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import InMemoryVectorStore


C:\Users\akash\AppData\Local\Temp\ipykernel_19300\942800745.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [16]:
loader = PyMuPDFLoader(file_path="../data/CorporatePolicyHandbook.pdf")
docs = loader.load()


In [17]:
splitter = RecursiveCharacterTextSplitter(chunk_size=600,chunk_overlap=50)
docs = splitter.split_documents(docs)

In [18]:
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1462.79it/s]


In [19]:
vector_store = InMemoryVectorStore.from_documents(
    embedding=embeddings,
    documents=docs
)

In [20]:
llm = ChatOpenRouter(
    api_key=api_key,
    model=model
)

In [ ]:
query = "Is there any crietria for creating User credentials"
chunks = vector_store.similarity_search(query)
# print(chunks[0].page_content)

context = ""

for chunk in chunks:
    context += chunk.page_content + "\n\n"

print(context)

In [25]:
prompt = f""" 

 You are Smart Rag system. You always provide response for user question based on the provide context.

 Question : {query}


 Context : {context}

"""

res = llm.invoke(prompt)

In [27]:
print(res.content)

**Yes**, according to the provided context, there are specific criteria for creating user credentials.

Based on the section **"2.3 Identity Governance, Credentialing & Password Requirements"**, the following criteria apply:

1. **Password Length:** Passwords must be **at least 14 characters long**.

2. **Password Complexity:** Passwords must combine **uppercase characters, lowercase characters, numerical digits, and approved special symbols**.

3. **Password Reuse:** Passwords must **not be reused across three consecutive cycles** or **shared across external consumer accounts**.

4. **Multi-Factor Authentication (MFA):** **MFA is strictly mandatory** across all enterprise access gateways.

5. **High-Privilege Access:** **Physical FIDO2 security keys** are required for high-privilege system administrators.

Additionally, the context notes that user credentials represent the **primary defensive line of enterprise infrastructure**, underscoring the importance of these requirements.

Woul